In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers.parent_document_retriever import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model_name= "gpt-3.5-turbo", max_tokens=500)

In [4]:
# Carregar o PDF
pdf_link = "Clean Architecture.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()
len(pages)


381

In [5]:
# Splitters para separação do documento em chunks

child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 200,
    length_function = len,
    add_start_index = True
)

In [6]:
# Storages -> Chroma para o recuperar os documentos relevantes e 
# InMemory fica o documento pai pois só precisa para enviar junto para o modelo

memoryStore = InMemoryStore()
vectorstore = Chroma(embedding_function=embeddings, persist_directory='db/childVectorDB')  

In [7]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=memoryStore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

batch_size = 20 # Número de documentos a serem adicionados por vez
for i in range(0, len(pages), batch_size):
    batch = pages[i:i + batch_size]
    parent_document_retriever.add_documents(batch, ids=None)
    print(f"Adicionados {len(batch)} documentos. Total processado: {i + len(batch)}/{len(pages)}")

print("Todos os documentos foram adicionados.")

Adicionados 20 documentos. Total processado: 20/381
Adicionados 20 documentos. Total processado: 40/381
Adicionados 20 documentos. Total processado: 60/381
Adicionados 20 documentos. Total processado: 80/381
Adicionados 20 documentos. Total processado: 100/381
Adicionados 20 documentos. Total processado: 120/381
Adicionados 20 documentos. Total processado: 140/381
Adicionados 20 documentos. Total processado: 160/381
Adicionados 20 documentos. Total processado: 180/381
Adicionados 20 documentos. Total processado: 200/381
Adicionados 20 documentos. Total processado: 220/381
Adicionados 20 documentos. Total processado: 240/381
Adicionados 20 documentos. Total processado: 260/381
Adicionados 20 documentos. Total processado: 280/381
Adicionados 20 documentos. Total processado: 300/381
Adicionados 20 documentos. Total processado: 320/381
Adicionados 20 documentos. Total processado: 340/381
Adicionados 20 documentos. Total processado: 360/381
Adicionados 20 documentos. Total processado: 380/3

In [9]:
parent_document_retriever.vectorstore.get()

{'ids': ['f7482b3a-7622-497e-bc3f-6f5a65a466a0',
  'e9c63992-8aa4-49d2-91b5-57755a9af52c',
  '0ab8368c-5828-4c8e-9174-e04d390817ef',
  '053da92c-4fa4-4da6-9fcf-a80cf14404bb',
  '2f062ec2-36d3-41f0-ba1f-9343433903ed',
  'ecce8bc5-38be-4bbd-b291-9d2dbc00df6d',
  '255d7b9c-6333-45f6-a41a-b1ea0ea8bdc0',
  'f1d5c671-fe6e-4b97-84a5-6525b86714d2',
  'f82c4c31-d84e-40af-85ae-67ec6ff86f03',
  '1a62d0e9-1816-48fa-b6e1-934919fb3aa7',
  '030c088e-c27d-4b1a-beb3-574dd073194a',
  'b5bcdfc6-8327-4bd7-8fc6-35a74ec6d475',
  '3b97bc83-b21e-46ee-8390-7bcd2e50d17c',
  '9bd0e4dd-ae9a-497f-83af-777e3d9a2018',
  '20ab5665-bd66-4ccb-a4c1-95ebb6db6e02',
  '409a8f5c-0b90-4531-bf4e-8036f536ed4d',
  '2038f285-bec9-49ba-b70c-b39450b8c51f',
  'c80ba21e-5a07-4285-8e1d-4ded747baef0',
  'cbad1679-ea7f-4dd9-972b-a59441299cf2',
  'eddcb76c-98ce-4eb5-8a10-0e2ea5b5f0e8',
  '242d5f76-a85b-4425-9b10-dcb91bef5963',
  '49a5ab74-6236-4d1a-b572-8fb10627881b',
  'cd2a2c2c-e74e-4b39-937f-9583c1b2abec',
  '5aaf13f5-f3e5-4ce3-bcd1-

In [19]:
TEMPLATE = """"
    Você é um desenvolvedor especialista com habilidades em engenharia e arquitetura do software. 
    Responda a pergunta abaixo utilizando o contexto informado e traduza a resposta para português.

    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [20]:
setup_retrival = RunnableParallel({"question": RunnablePassthrough(), "context": parent_document_retriever})

output_parser = StrOutputParser()

In [21]:
parent_chain_retrieval = setup_retrival | rag_prompt | llm | output_parser

In [22]:
parent_document_retriever.invoke("Como utilizar a camada Application em uma arquitetura?")

[Document(metadata={'producer': 'calibre 3.7.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.7.0 [https://calibre-ebook.com]', 'creationdate': '2017-09-15T12:39:59+00:00', 'author': 'Robert C. Martin', 'moddate': '2018-01-31T20:00:46+08:00', 'pxcviewerinfo': "PDF-XChange Viewer;2.5.313.1;Jun  8 2015;11:48:33;D:20170920170036+08'00'", 'title': "Clean Architecture: A Craftsman's Guide to Software Structure and Design (Robert C. Martin Series)", 'source': 'Clean Architecture.pdf', 'total_pages': 429, 'page': 191, 'page_label': '192', 'start_index': 0}, page_content='the library. That architecture would scream: “LIBRARY.”\nSo what does the architecture of your application scream? When you look at the top-\nlevel directory structure, and the source files in the highest-level package, do they\nscream “Health Care System,” or “Accounting System,” or “Inventory Management\nSystem”? Or do they scream “Rails,” or “Spring/Hibernate,” or “ASP”?\nT\nHE\n T\nHEME OF AN\n A\nRCHITECTURE\nGo bac